In [2]:
!pip install chromadb sentence-transformers groq pypdf pandas scikit-learn ipywidgets --user

In [3]:
import pandas as pd
import chromadb
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from groq import Groq
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load dataset
df = pd.read_csv("Sample - Superstore.csv", encoding='latin1')
print("Dataset loaded ✅")
print(f"Shape: {df.shape}")
df.head()

Dataset loaded ✅
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
texts = []
for _, row in df.iterrows():
    text = f"Order {row['Order ID']} from {row['City']}, {row['State']}. Category: {row['Category']}, Sub-Category: {row['Sub-Category']}. Product: {row['Product Name']}. Sales: ${row['Sales']:.2f}, Profit: ${row['Profit']:.2f}."
    texts.append(text)

print(f"Total chunks created: {len(texts)} ✅")
print(f"\nExample chunk:")
print(texts[0])

Total chunks created: 9994 ✅

Example chunk:
Order CA-2016-152156 from Henderson, Kentucky. Category: Furniture, Sub-Category: Bookcases. Product: Bush Somerset Collection Bookcase. Sales: $261.96, Profit: $41.91.


In [5]:
# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("superstore")

# Store in batches
batch_size = 500
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    embeddings = embedding_model.encode(batch).tolist()
    ids = [str(j) for j in range(i, i+len(batch))]
    collection.add(documents=batch, embeddings=embeddings, ids=ids)
    print(f"Stored {i+len(batch)} chunks...")

print("ChromaDB ready ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored 500 chunks...
Stored 1000 chunks...
Stored 1500 chunks...
Stored 2000 chunks...
Stored 2500 chunks...
Stored 3000 chunks...
Stored 3500 chunks...
Stored 4000 chunks...
Stored 4500 chunks...
Stored 5000 chunks...
Stored 5500 chunks...
Stored 6000 chunks...
Stored 6500 chunks...
Stored 7000 chunks...
Stored 7500 chunks...
Stored 8000 chunks...
Stored 8500 chunks...
Stored 9000 chunks...
Stored 9500 chunks...
Stored 9994 chunks...
ChromaDB ready ✅


In [6]:
# Get your FREE Groq API key from console.groq.com
# Replace below with your own key
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq API ready ✅")

Groq API ready ✅


In [7]:
def retrieve(question, k=4):
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )
    return results['documents'][0]

def rag_answer(question):
    chunks = retrieve(question)
    context = "\n".join(chunks)
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """You are an expert data analyst assistant.
Answer ONLY from the context provided.
Give clear structured business insights.
If answer not in context say 'I don't have that information.'"""},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content

print("RAG functions ready ✅")

RAG functions ready ✅


In [16]:
def data_analysis_summary():
    print("=" * 55)
    print("        SUPERSTORE SALES ANALYSIS SUMMARY")
    print("=" * 55)
    print(f"Total Sales:   ${df['Sales'].sum():,.2f}")
    print(f"Total Profit:  ${df['Profit'].sum():,.2f}")
    print(f"Total Orders:  {df['Order ID'].nunique()}")
    print(f"Total Products:{df['Product Name'].nunique()}")

    print("\nTop 5 Profitable Products:")
    top_products = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head()
    print(top_products.to_string())

    print("\nSales by Category:")
    category_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
    print(category_sales.to_string())

    print("\nTop 5 Cities by Sales:")
    top_cities = df.groupby('City')['Sales'].sum().sort_values(ascending=False).head()
    print(top_cities.to_string())
    print("=" * 55)

data_analysis_summary()

        SUPERSTORE SALES ANALYSIS SUMMARY
Total Sales:   $2,297,200.86
Total Profit:  $286,397.02
Total Orders:  5009
Total Products:1850

Top 5 Profitable Products:
Product Name
Canon imageCLASS 2200 Advanced Copier                                          25199.9280
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind     7753.0390
Hewlett Packard LaserJet 3310 Copier                                            6983.8836
Canon PC1060 Personal Laser Copier                                              4570.9347
HP Designjet T520 Inkjet Large Format Printer - 24" Color                       4094.9766

Sales by Category:
Category
Technology         836154.0330
Furniture          741999.7953
Office Supplies    719047.0320

Top 5 Cities by Sales:
City
New York City    256368.161
Los Angeles      175851.341
Seattle          119540.742
San Francisco    112669.092
Philadelphia     109077.013


In [8]:
# Prepare data
X = df[['Sales', 'Quantity', 'Discount']].values
y = df['Profit'].values

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
ml_model = LinearRegression()
ml_model.fit(X_train, y_train)

# Accuracy
predictions = ml_model.predict(X_test)
error = mean_absolute_error(y_test, predictions)
print(f"ML Model trained ✅")
print(f"Average prediction error: ${error:.2f}")

def predict_profit(sales, quantity, discount):
    input_data = np.array([[sales, quantity, discount]])
    predicted_profit = ml_model.predict(input_data)[0]
    print(f"\nInput:")
    print(f"  Sales:    ${sales}")
    print(f"  Quantity: {quantity}")
    print(f"  Discount: {discount*100}%")
    print(f"Predicted Profit: ${predicted_profit:.2f}")

# Test predictions
predict_profit(500, 3, 0.1)
predict_profit(500, 3, 0.5)
predict_profit(2000, 10, 0.0)

ML Model trained ✅
Average prediction error: $69.77

Input:
  Sales:    $500
  Quantity: 3
  Discount: 10.0%
Predicted Profit: $118.72

Input:
  Sales:    $500
  Quantity: 3
  Discount: 50.0%
Predicted Profit: $35.18

Input:
  Sales:    $2000
  Quantity: 10
  Discount: 0.0%
Predicted Profit: $483.03


In [9]:
print("=" * 55)
print("    AI-POWERED SUPERSTORE SALES ANALYST")
print("=" * 55)

input_box = widgets.Text(
    placeholder="Ask a business question...",
    layout=widgets.Layout(width="70%")
)

button = widgets.Button(
    description="Ask AI",
    button_style="primary"
)

output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        question = input_box.value
        if question:
            print(f"Q: {question}")
            print("Thinking...")
            answer = rag_answer(question)
            print(f"\nA: {answer}")
            print("-" * 50)

button.on_click(on_click)
display(widgets.HBox([input_box, button]))
display(output)

    AI-POWERED SUPERSTORE SALES ANALYST


Output()

In [10]:
print("=" * 55)
print("        PROJECT COMPLETE ✅")
print("=" * 55)
print("Project: AI-Powered Superstore Sales Analyst")
print("=" * 55)
print("Tech Stack:")
print("  • Python          → Core language")
print("  • Pandas          → Data analysis")
print("  • ChromaDB        → Vector database")
print("  • Sentence-Trans  → Text embeddings")
print("  • Groq LLM        → AI answers")
print("  • Scikit-learn    → ML predictions")
print("  • ipywidgets      → Chat interface")
print("=" * 55)
print("Features:")
print("  ✅ Natural language querying")
print("  ✅ Semantic vector search")
print("  ✅ AI business insights")
print("  ✅ Sales data analysis")
print("  ✅ ML profit prediction")
print("=" * 55)
print("Dataset: Superstore Sales — 9994 records")
print("=" * 55)

        PROJECT COMPLETE ✅
Project: AI-Powered Superstore Sales Analyst
Tech Stack:
  • Python          → Core language
  • Pandas          → Data analysis
  • ChromaDB        → Vector database
  • Sentence-Trans  → Text embeddings
  • Groq LLM        → AI answers
  • Scikit-learn    → ML predictions
  • ipywidgets      → Chat interface
Features:
  ✅ Natural language querying
  ✅ Semantic vector search
  ✅ AI business insights
  ✅ Sales data analysis
  ✅ ML profit prediction
Dataset: Superstore Sales — 9994 records


In [11]:
readme = """# AI-Powered Superstore Sales Analyst

## Project Overview
An end-to-end AI-powered business analyst tool that enables
natural language querying over 10,000+ sales records using
a RAG (Retrieval Augmented Generation) pipeline.

## Tech Stack
- Python
- Pandas
- ChromaDB (Vector Database)
- Sentence-Transformers (Embeddings)
- Groq LLM (Free AI) — Get free key at console.groq.com
- Scikit-learn (ML Predictions)
- ipywidgets (Chat Interface)

## Features
- Natural language querying over sales data
- Semantic vector search using ChromaDB
- AI powered business insights using Groq LLM
- Sales data analysis using Pandas
- ML profit prediction using Scikit-learn

## How to Run
1. Install requirements:
   pip install -r requirements.txt
2. Get FREE Groq API key from console.groq.com
3. Add your key in Cell 5:
   os.environ["GROQ_API_KEY"] = "your-key-here"
4. Run all cells in order

## Output Screenshots

### Data Analysis Summary
![Analysis](screenshots/screenshot5.png)

### AI Question Answers
![Profitable Products](screenshots/screenshot1.png)
![Regional Sales](screenshots/screenshot2.png)
![Category Revenue](screenshots/screenshot3.png)
![Negative Profit](screenshots/screenshot4.png)

### ML Profit Prediction
![ML Prediction](screenshots/screenshot6.png)

## Sample Questions You Can Ask
- What are the most profitable products?
- Which city has the highest sales?
- What is the best performing category?
- Which region has lowest profit?

## ML Prediction
Predict profit based on:
- Sales amount
- Quantity
- Discount percentage

## Dataset
- Name: Superstore Sales Dataset
- Records: 9994
- Source: Kaggle
- Link: kaggle.com/datasets/vivek468/superstore-dataset-final

## Author
Built as part of Data Analysis + AI portfolio project.
"""

with open("README.md", "w") as f:
    f.write(readme)
print("README created ✅")

README created ✅


In [12]:
requirements = """chromadb
sentence-transformers
groq
pypdf
pandas
scikit-learn
ipywidgets
numpy
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt created ✅")

requirements.txt created ✅


In [13]:
import os
os.makedirs("screenshots", exist_ok=True)
print("Screenshots folder created ✅")

Screenshots folder created ✅
